<a href="https://colab.research.google.com/github/m-manuelmussa/Drug-Repurposing-IPEs-MtrD-Pharmacophores-Molecular-Docking-Dynamics/blob/main/2_Curadoria_Qu%C3%ADmica_%26_Morgan_Fingerprints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Curadoria Química & Morgan Fingerprints**
**Objectivo**: Curar fármacos seleccionados no dataset final e calcular fingerprints



***Autor: Micliete Lopes Manuel Mussa***

## **1. Instalação de Dependências**

In [1]:
!pip install rdkit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 38.3 MB/s eta 0:00:00


## **2. Importação de bibliotecas**

In [2]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger

## **3. Leitura do dataset bruto**

In [3]:
# 1. Carrega o DataFrame salvo na etapa anterior
input_file = "medicamentos_aprovados_chembl_curado.csv"
df = pd.read_csv(input_file)
print(f"Total de registros lidos do CSV: {len(df)}")

Total de registros lidos do CSV: 3310


## **4. Curação química rigorosa**

In [4]:
# 2. Configura os parâmetros de tautômeros e inicializa o enumerador
params = rdMolStandardize.CleanupParameters()
params.maxTautomers = 50  # Define o limite máximo de tautômeros nos parâmetros

tautomer_enumerator = rdMolStandardize.TautomerEnumerator(params)

def pipeline_curadoria_quimica(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    try:
        # A. Cleanup geral e remoção de sais/contraíons
        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)

        # B. Filtro de Inorgânicos: Remove se não contiver carbono ou for muito pequeno
        has_carbon = any(atom.GetAtomicNum() == 6 for atom in mol.GetAtoms())
        if not has_carbon or mol.GetNumHeavyAtoms() < 2:
            return None

        # C. Tautômero canônico
        mol = tautomer_enumerator.Canonicalize(mol)

        # D. Retorna o SMILES canônico limpo
        return Chem.MolToSmiles(mol, canonical=True)

    except Exception:
        return None

# 3. Aplica a curadoria química
df['smiles_curado'] = df['smiles'].apply(pipeline_curadoria_quimica)

# 4. Remove falhas e duplicatas pós-padronização
df_quimico = df.dropna(subset=['smiles_curado']).copy()

duplicatas_removidas = len(df_quimico) - len(df_quimico.drop_duplicates(subset=['smiles_curado']))
df_quimico = df_quimico.drop_duplicates(subset=['smiles_curado'], keep='first').reset_index(drop=True)

Streaming output truncated to the last 5000 lines.
[09:47:37] New largest fragment: CCc1nc(C(N)=O)c(Nc2ccc(N3CCC(N4CCN(C)CC4)CC3)c(OC)c2)nc1NC1CCOCC1 (84)
[09:47:37] Fragment: O=C(O)/C=C/C(=O)O
[09:47:38] Can't kekulize mol.  Unkekulized atoms: 4 32
[09:47:38] Can't kekulize mol.  Unkekulized atoms: 4 32
[09:47:38] Initializing MetalDisconnector
[09:47:38] Running MetalDisconnector
[09:47:38] Initializing Normalizer
[09:47:38] Running Normalizer
[09:47:38] Initializing MetalDisconnector
[09:47:38] Running MetalDisconnector
[09:47:38] Initializing Normalizer
[09:47:38] Running Normalizer
[09:47:38] Running LargestFragmentChooser
[09:47:38] Fragment: O
[09:47:38] New largest fragment: O (3)
[09:47:38] Fragment: O=C(NC1CCNCC1)[C@@H]1CC[C@@H]2CN1C(=O)N2OS(=O)(=O)O
[09:47:38] New largest fragment: O=C(NC1CCNCC1)[C@@H]1CC[C@@H]2CN1C(=O)N2OS(=O)(=O)O (43)
[09:47:38] Initializing MetalDisconnector
[09:47:38] Running MetalDisconnector
[09:47:38] Initializing Normalizer
[09:47:38] Running Normal

In [5]:
print(f"Registros iniciais: {len(df)}")
print(f"Registros inválidos/inorgânicos descartados: {len(df) - len(df.dropna(subset=['smiles_curado']))}")
print(f"Duplicatas estruturais removidas: {duplicatas_removidas}")
print(f"TOTAL FINAL DE FÁRMACOS CURADOS: {len(df_quimico)}")

Registros iniciais: 3310
Registros inválidos/inorgânicos descartados: 76
Duplicatas estruturais removidas: 893
TOTAL FINAL DE FÁRMACOS CURADOS: 2341


## **5. Cálculo de Morgan Fingerprints**

In [6]:
# 1. Função para gerar o Morgan Fingerprint em Bit String
def gerar_morgan_fp_string(smiles, radius=2, nBits=2048):
    """Gera o Morgan Fingerprint (ECFP4) em formato de Bit String."""
    if not isinstance(smiles, str) or not smiles.strip():
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Gera o ExplicitBitVect (ECFP4 com raio 2 e 2048 bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    return fp.ToBitString()

In [7]:
# 2. Calcula os Morgan Fingerprints utilizando a variável df_quimico já carregada
print("Gerando Morgan Fingerprints (ECFP4, 2048 bits) a partir da variável 'df_quimico'...")
df_quimico['morgan_fp'] = df_quimico['smiles_curado'].apply(gerar_morgan_fp_string)

# 3. Garante a remoção de eventuais falhas
df_quimico = df_quimico.dropna(subset=['morgan_fp']).reset_index(drop=True)

Gerando Morgan Fingerprints (ECFP4, 2048 bits) a partir da variável 'df_quimico'...


[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerator
[09:50:58] DEPRECATION WARNING: please use MorganGenerat

In [17]:
print(f"Total de fármacos com fingerprints gerados: {len(df_quimico)}")

Total de fármacos com fingerprints gerados: 2413


In [8]:
# Exibe as primeiras linhas com o fingerprint
df_quimico[['chembl_id', 'nome_preferencial', 'smiles_curado', 'morgan_fp']].head()

,chembl_id,nome_preferencial,smiles_curado,morgan_fp
0,CHEMBL2,PRAZOSIN,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC,0000000000000000000000000000000000000000000000...
1,CHEMBL3,NICOTINE,CN1CCC[C@H]1c1cccnc1,0000000000000000000000000000000000000000000000...
2,CHEMBL4,OFLOXACIN,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23,0000000000000010000000000000000000000000000000...
3,CHEMBL5,NALIDIXIC ACID,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21,0000000000000000000000000000000000100000000000...
4,CHEMBL6,INDOMETHACIN,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1,0000000000000000000000000000000000000000000001...


## **6. Exportação do Dataset Quimicamente Curado e Fingerprints Calculados**

In [9]:
# 4. Salva o CSV final atualizado com a coluna morgan_fp
output_file = "medicamentos_aprovados_chembl_curadoria_quimica.csv"
df_quimico.to_csv(output_file, index=False, encoding='utf-8')
print(f"Arquivo CSV final salvo com sucesso")

Arquivo CSV final salvo com sucesso


***Autor: Micliete Lopes Manuel Mussa, Farmacêutico.***